# Fashion E-commerce Analytics — Data Transformation (Star Schema)

## Objective

Transform the cleaned datasets (`data/cleaned/`) into a dimensional model
(star schema) ready to be loaded into SQL Server:

```
                 Dim_Customer
                      |
Dim_Date ──────── Fact_Sales ──────── Dim_Product
                      |
                 Dim_Store ── Dim_Employee
```

`Dim_Discount` is kept as a standalone reference table (promotional
periods per category), joined on-demand for promo analysis (Phase 9) —
not as a direct FK on `Fact_Sales`, since the discount **rate** already
lives on each transaction line.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)


In [2]:
BASE_DIR = Path("..")

CLEANED_DIR = BASE_DIR / "data" / "cleaned"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load cleaned datasets

In [3]:
customers = pd.read_csv(CLEANED_DIR / "customers_clean.csv", dtype={"Telephone": "string"})
products = pd.read_csv(CLEANED_DIR / "products_clean.csv")
discounts = pd.read_csv(CLEANED_DIR / "discounts_clean.csv", parse_dates=["Start", "End"])
employees = pd.read_csv(CLEANED_DIR / "employees_clean.csv")
stores = pd.read_csv(CLEANED_DIR / "stores_clean.csv")

transactions = pd.read_csv(
    CLEANED_DIR / "transactions_clean.csv",
    parse_dates=["Date"]
)

for name, df in {
    "customers": customers, "products": products, "discounts": discounts,
    "employees": employees, "stores": stores, "transactions": transactions
}.items():
    print(f"{name}: {df.shape}")


customers: (1643306, 9)
products: (17940, 12)
discounts: (181, 6)
employees: (404, 4)
stores: (35, 8)
transactions: (6416029, 20)


## 2. Fix carried over from `03_data_cleaning.ipynb`

Redo the duplicate removal and `Return Ratio` computation defensively —
harmless if `transactions_clean.csv` already had them applied
(`drop_duplicates` is idempotent, and we only (re)compute `Return Ratio`
if it's missing).

In [4]:
before = len(transactions)
transactions = transactions.drop_duplicates().reset_index(drop=True)
print(f"Transactions: {before} -> {len(transactions)} (removed {before - len(transactions)} duplicates)")


Transactions: 6416029 -> 6416029 (removed 0 duplicates)


In [5]:
if "Return Ratio" not in transactions.columns:
    transactions["Return Ratio"] = 0.0

    return_mask = transactions["Transaction Type"] == "Return"

    transactions.loc[return_mask, "Return Ratio"] = (
        transactions.loc[return_mask, "Line Total"].abs()
        / (
            transactions.loc[return_mask, "Unit Price"]
            * transactions.loc[return_mask, "Quantity"]
        )
    )
    print("Return Ratio computed.")
else:
    print("Return Ratio already present.")


Return Ratio already present.


## 3. Dim_Date

In [6]:
transactions["Date"] = pd.to_datetime(transactions["Date"])
transactions["Sale Date"] = transactions["Date"].dt.date

min_date = transactions["Date"].dt.date.min()
max_date = transactions["Date"].dt.date.max()

dim_date = pd.DataFrame({"Date": pd.date_range(min_date, max_date, freq="D")})

dim_date["Date_Key"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month_Name"] = dim_date["Date"].dt.month_name()
dim_date["Day"] = dim_date["Date"].dt.day
dim_date["Day_Of_Week"] = dim_date["Date"].dt.dayofweek
dim_date["Day_Name"] = dim_date["Date"].dt.day_name()
dim_date["Is_Weekend"] = dim_date["Day_Of_Week"].isin([5, 6])
dim_date["Week_Of_Year"] = dim_date["Date"].dt.isocalendar().week.astype(int)

dim_date = dim_date[
    ["Date_Key", "Date", "Year", "Quarter", "Month", "Month_Name",
     "Day", "Day_Of_Week", "Day_Name", "Is_Weekend", "Week_Of_Year"]
]

print(dim_date.shape)
dim_date.head()


(808, 11)


,Date_Key,Date,Year,Quarter,Month,Month_Name,Day,Day_Of_Week,Day_Name,Is_Weekend,Week_Of_Year
0,20230101,2023-01-01,2023,1,1,January,1,6,Sunday,True,52
1,20230102,2023-01-02,2023,1,1,January,2,0,Monday,False,1
2,20230103,2023-01-03,2023,1,1,January,3,1,Tuesday,False,1
3,20230104,2023-01-04,2023,1,1,January,4,2,Wednesday,False,1
4,20230105,2023-01-05,2023,1,1,January,5,3,Thursday,False,1


## 4. Dim_Customer

In [7]:
current_year = pd.Timestamp.today().year

dim_customer = customers.rename(columns={
    "Customer ID": "Customer_Key",
    "Name": "Customer_Name",
    "Job Title": "Job_Title",
    "Date Of Birth": "Date_Of_Birth"
}).copy()

dim_customer["Date_Of_Birth"] = pd.to_datetime(dim_customer["Date_Of_Birth"], errors="coerce")
dim_customer["Age"] = current_year - dim_customer["Date_Of_Birth"].dt.year

print(dim_customer.shape)
dim_customer.head()


(1643306, 10)


,Customer_Key,Customer_Name,Email,Telephone,City,Country,Gender,Date_Of_Birth,Job_Title,Age
0,1,Tyler Garcia,tyler.garcia@fake_gmail.com,922.970.2265x47563,New York,United States,M,2003-07-15,Unknown,23
1,2,Joshua Miller,joshua.miller@fake_gmail.com,+1-958-729-6169,New York,United States,M,2000-06-16,Records manager,26
2,3,Alison Marshall DDS,alison.marshall.dds@fake_hotmail.com,+1-645-567-0876x5409,New York,United States,F,2003-07-22,Unknown,23
3,4,Jeffery Acosta,jeffery.acosta@fake_yahoo.com,212.336.0912x84994,New York,United States,M,1996-11-12,Proofreader,30
4,5,Ashley Sanders,ashley.sanders@fake_hotmail.com,7814535781,New York,United States,F,1998-02-10,Exercise physiologist,28


## 5. Dim_Product

Only the English description is kept for BI use; the other language columns are dropped here (still available in `products_clean.csv` if ever needed).

In [8]:
dim_product = products.rename(columns={
    "Product ID": "Product_Key",
    "Sub Category": "Sub_Category",
    "Description EN": "Description",
    "Sizes": "Available_Sizes",
    "Production Cost": "Production_Cost"
})[[
    "Product_Key", "Category", "Sub_Category", "Description",
    "Color", "Available_Sizes", "Production_Cost"
]].copy()

print(dim_product.shape)
dim_product.head()


(17940, 7)


,Product_Key,Category,Sub_Category,Description,Color,Available_Sizes,Production_Cost
0,1,Feminine,Coats and Blazers,Sports Velvet Sports With Buttons,Unknown,S|M|L|XL,10.73
1,2,Feminine,Sweaters and Knitwear,Luxurious Pink Denim With Buttons,PINK,S|M|L|XL,19.55
2,3,Feminine,Dresses and Jumpsuits,Black Tricot Printed Tricot,BLACK,S|M|L|XL,25.59
3,4,Feminine,Shirts and Blouses,Basic Cotton Blouse,Unknown,S|M|L|XL,27.62
4,5,Feminine,T-shirts and Tops,Basic Cotton T-Shirt,Unknown,S|M|L,11.69


## 6. Dim_Store

In [9]:
dim_store = stores.rename(columns={
    "Store ID": "Store_Key",
    "Store Name": "Store_Name",
    "Number of Employees": "Number_Of_Employees",
    "ZIP Code": "ZIP_Code"
})[[
    "Store_Key", "Store_Name", "City", "Country",
    "ZIP_Code", "Latitude", "Longitude", "Number_Of_Employees"
]].copy()

print(dim_store.shape)
dim_store.head()


(35, 8)


,Store_Key,Store_Name,City,Country,ZIP_Code,Latitude,Longitude,Number_Of_Employees
0,1,Store New York,New York,United States,10001,40.7128,-74.0060,10
1,2,Store Los Angeles,Los Angeles,United States,90001,34.0522,-118.2437,8
2,3,Store Chicago,Chicago,United States,60601,41.8781,-87.6298,9
3,4,Store Houston,Houston,United States,77001,29.7604,-95.3698,10
4,5,Store Phoenix,Phoenix,United States,85001,33.4484,-112.0740,9


## 7. Dim_Employee

In [10]:
dim_employee = employees.rename(columns={
    "Employee ID": "Employee_Key",
    "Store ID": "Store_Key",
    "Name": "Employee_Name",
    "Position": "Position"
})[["Employee_Key", "Store_Key", "Employee_Name", "Position"]].copy()

print(dim_employee.shape)
dim_employee.head()


(404, 4)


,Employee_Key,Store_Key,Employee_Name,Position
0,1,1,Stephen Johnson,Store Manager
1,2,1,Rebecca Myers,Assistant Manager
2,3,1,Katherine Buchanan,Cashier
3,4,1,Jessica Hicks,Stock Clerk
4,5,1,Ryan Gross,Sales Associate


## 8. Dim_Discount

Standalone reference table — one row per promotional period/category. Not joined into `Fact_Sales`.

In [11]:
dim_discount = discounts.rename(columns={
    "Discont": "Discount_Rate",
    "Sub Category": "Sub_Category"
}).copy()

dim_discount["Discount_Key"] = np.arange(1, len(dim_discount) + 1)

dim_discount = dim_discount[
    ["Discount_Key", "Category", "Sub_Category", "Start", "End", "Discount_Rate", "Description"]
]

print(dim_discount.shape)
dim_discount.head()


(181, 7)


,Discount_Key,Category,Sub_Category,Start,End,Discount_Rate,Description
0,1,Feminine,Coats and Blazers,2020-01-01,2020-01-10,0.4,40% discount during our New Year Winter Sale
1,2,Feminine,Sweaters and Knitwear,2020-01-01,2020-01-10,0.4,40% discount during our New Year Winter Sale
2,3,Masculine,Coats and Blazers,2020-01-01,2020-01-10,0.4,40% discount during our New Year Winter Sale
3,4,Masculine,Sweaters and Sweatshirts,2020-01-01,2020-01-10,0.4,40% discount during our New Year Winter Sale
4,5,Children,Coats,2020-01-01,2020-01-10,0.4,40% discount during our New Year Winter Sale


## 9. Fact_Sales

Grain: one row per invoice line (`Invoice ID` + `Line`).

In [12]:
fact_sales = transactions.rename(columns={
    "Invoice ID": "Invoice_ID",
    "Customer ID": "Customer_Key",
    "Product ID": "Product_Key",
    "Store ID": "Store_Key",
    "Employee ID": "Employee_Key",
    "Unit Price": "Unit_Price",
    "Line Total": "Line_Total",
    "Invoice Total": "Invoice_Total",
    "Transaction Type": "Transaction_Type",
    "Payment Method": "Payment_Method",
    "Currency Symbol": "Currency_Symbol",
    "Return Ratio": "Return_Ratio"
}).copy()

fact_sales["Date_Key"] = fact_sales["Date"].dt.strftime("%Y%m%d").astype(int)

fact_sales = fact_sales[[
    "Invoice_ID", "Line", "Date_Key", "Customer_Key", "Product_Key",
    "Store_Key", "Employee_Key", "SKU", "Size", "Color",
    "Quantity", "Unit_Price", "Discount", "Line_Total", "Invoice_Total",
    "Return_Ratio", "Transaction_Type", "Payment_Method", "Currency"
]]

print(fact_sales.shape)
fact_sales.head()


(6416029, 19)


,Invoice_ID,Line,Date_Key,Customer_Key,Product_Key,Store_Key,Employee_Key,SKU,Size,Color,Quantity,Unit_Price,Discount,Line_Total,Invoice_Total,Return_Ratio,Transaction_Type,Payment_Method,Currency
0,INV-US-001-03558761,1,20230101,47162,485,1,7,MASU485-M-,M,Unknown,1,80.5,0.0,80.5,126.7,0.0,Sale,Cash,USD
1,INV-US-001-03558761,2,20230101,47162,2779,1,7,CHCO2779-G-,G,Unknown,1,31.5,0.4,18.9,126.7,0.0,Sale,Cash,USD
2,INV-US-001-03558761,3,20230101,47162,64,1,7,MACO64-M-NEUTRAL,M,NEUTRAL,1,45.5,0.4,27.3,126.7,0.0,Sale,Cash,USD
3,INV-US-001-03558762,1,20230101,10142,131,1,6,FECO131-M-BLUE,M,BLUE,1,70.0,0.4,42.0,77.0,0.0,Sale,Cash,USD
4,INV-US-001-03558762,2,20230101,10142,716,1,6,MAT-716-L-WHITE,L,WHITE,1,26.0,0.0,26.0,77.0,0.0,Sale,Cash,USD


## 10. Referential integrity checks

Any FK below that doesn't return 0 needs fixing before loading SQL Server — a bad FK will either break the load or force `NULL`s silently.

In [13]:
checks = {
    "Date_Key": (~fact_sales["Date_Key"].isin(dim_date["Date_Key"])).sum(),
    "Customer_Key": (~fact_sales["Customer_Key"].isin(dim_customer["Customer_Key"])).sum(),
    "Product_Key": (~fact_sales["Product_Key"].isin(dim_product["Product_Key"])).sum(),
    "Store_Key": (~fact_sales["Store_Key"].isin(dim_store["Store_Key"])).sum(),
    "Employee_Key": (~fact_sales["Employee_Key"].isin(dim_employee["Employee_Key"])).sum(),
}

for key, orphans in checks.items():
    print(f"{key}: {orphans} orphan rows")


Date_Key: 0 orphan rows
Customer_Key: 0 orphan rows
Product_Key: 0 orphan rows
Store_Key: 0 orphan rows
Employee_Key: 0 orphan rows


## 11. Export to `data/processed/`

In [14]:
tables = {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_store": dim_store,
    "dim_employee": dim_employee,
    "dim_discount": dim_discount,
    "fact_sales": fact_sales,
}

for name, df in tables.items():
    path = PROCESSED_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"{name}: {df.shape} -> {path}")


dim_date: (808, 11) -> ..\data\processed\dim_date.csv
dim_customer: (1643306, 10) -> ..\data\processed\dim_customer.csv
dim_product: (17940, 7) -> ..\data\processed\dim_product.csv
dim_store: (35, 8) -> ..\data\processed\dim_store.csv
dim_employee: (404, 4) -> ..\data\processed\dim_employee.csv
dim_discount: (181, 7) -> ..\data\processed\dim_discount.csv
fact_sales: (6416029, 19) -> ..\data\processed\fact_sales.csv
